In [3]:
"""A6 - per-recording evaluation of variants A and B (and C if B's file is given) against 3D mocap.
One time shift for all variants: lag = best_lag(2D angle with no visibility rule, 3D truth)."""
import numpy as np
from angles import joint_angle, ELBOW
from visibility import mask_low_visibility
from angles_world import elbow_angle_3d, mask_world_like
from benchmark_common import best_lag, overlap, truth_elbow_3d

ARMS = {"right": ("RightArm", "RightForeArm", "RightHand"),
        "left": ("LeftArm", "LeftForeArm", "LeftHand")}


def view_of(rec):
    if rec["cam"] == "17":
        return rec["orient"]
    return {"front": "side", "profile": "front", "side": "front"}.get(rec["orient"], rec["orient"])


def active_sides(rec):
    st = rec.get("subtype", "")
    if "right" in st or "left" in st:
        return [s for s in ("right", "left") if s in st]
    return ["right", "left"]


def evaluate_recording(rec, cache, truth3d, names, thr=0.70, max_lag=30, extra=None):
    """rec: dict(name, cam, ex, orient, subtype, f0w). cache: dict(lm, world, size).
    extra: optional {"C": {"right": angles over the same window}} from Person B."""
    lm, world, size = cache["lm"], cache["world"], cache["size"]
    f0 = rec["f0w"]
    out = []
    for side in active_sides(rec):
        other = "left" if side == "right" else "right"
        tl = {s: truth_elbow_3d(truth3d, names, ARMS[s])[f0: f0 + len(lm)] for s in ("right", "left")}
        align = joint_angle(mask_low_visibility(lm, 0.0), ELBOW[side], size)
        lag, r_named = best_lag(align, tl[side], max_lag)
        lag_o, r_other = best_lag(align, tl[other], max_lag)
        swapped = r_other > r_named + 0.1
        truth, lag = (tl[other], lag_o) if swapped else (tl[side], lag)

        masked = mask_low_visibility(lm, thr)
        preds = {"A": joint_angle(masked, ELBOW[side], size),
                 "B": elbow_angle_3d(mask_world_like(world, masked), side)}
        for name, per_side in (extra or {}).items():
            if side in per_side:
                preds[name] = np.asarray(per_side[side], float)[: len(lm)]
        aligned, y = {}, None
        for name, p in preds.items():
            x, y_ = overlap(p, truth, lag)
            aligned[name], y = x, y_
        out.append(dict(side=side, swapped=bool(swapped), lag=int(lag), view=view_of(rec), preds=aligned, truth=y))
    return out

In [11]:
subprocess.run(["git","clone","-q","-b","chamodi-week2",
                "https://github.com/suvini2001/AI-based-Physiotherapy-project-.git", REPO_DIR], check=True)
fetch("3d_joints.zip"); unzip("3d_joints.zip", "3d_joints")   # confirm the file name on the Zenodo page

NameError: name 'subprocess' is not defined

In [12]:
import os
import re
import pandas as pd

# 1. Set the dataset directory path
DATA = "data"  # Change this to "/content/data" if using Google Colab

# 2. Re-create or load the 'mine' list of 68 paired recordings
# Option A: If you saved your paired metadata CSV from Step 5:
mine = pd.read_csv("paired_video_joint_metadata.csv").to_dict(
    orient="records"
)

# Option B: If your DataFrame from Step 5 is named 'pairs_df' or 'df' in memory:
# mine = pairs_df.to_dict(orient='records')


# 3. Define helper functions for matching 3D files to 2D recordings
def file_key(path):
    """Extracts a normalized identifier from a file path."""
    base = os.path.basename(str(path)).rsplit(".", 1)[0].lower()
    base = base.replace("-transposed", "").replace("_", "-")
    return base


def same_recording(k1, k2):
    """Checks if two file keys correspond to the same recording session."""
    # Extract participant/camera pattern (e.g. 'pm-000-c17')
    m1 = re.search(r"pm[_\-]?\d+[\-_]c\d+", k1)
    m2 = re.search(r"pm[_\-]?\d+[\-_]c\d+", k2)
    if m1 and m2:
        return m1.group(0) == m2.group(0)
    return k1 == k2 or k1 in k2 or k2 in k1

FileNotFoundError: [Errno 2] No such file or directory: 'paired_video_joint_metadata.csv'

In [13]:
import glob
J3D = sorted(p for p in glob.glob(f"{DATA}/3d_joints/**/*", recursive=True) if p.lower().endswith((".npy", ".csv", ".txt", ".npz")))
print(len(J3D), "3D files; examples:", [os.path.basename(p) for p in J3D[:4]])
def find3d(rec):
    rk = file_key(rec["name"])
    js = [p for p in J3D if same_recording(rk, file_key(p))]
    js30 = [p for p in js if "30fps" in os.path.basename(p).lower()]
    js = js30 or js
    return js[0] if len(js) == 1 else None
missing = [r["name"] for r in mine if find3d(r) is None]
print("recordings with no unique 3D file:", missing[:10], "(0 is what you want)")

0 3D files; examples: []


NameError: name 'mine' is not defined

In [14]:
rows = {"A": [], "B": []}
records_for_curve = []
for n, rec in enumerate(mine, 1):
    path3d, wpath = find3d(rec), world_path(rec)
    if path3d is None or not os.path.exists(wpath):
        print("skip", rec["name"]); continue
    z = np.load(wpath)
    cache = dict(lm=z["lm"], world=z["world"], size=tuple(int(v) for v in z["size"]))
    t3 = load_truth_array(path3d, NAMES)
    if abs(len(t3) / video_frames(rec["video"]) - 1) > 0.02:
        t3 = t3[np.round(np.linspace(0, len(t3) - 1, video_frames(rec["video"]))).astype(int)]
    for o in evaluate_recording(rec, cache, t3, NAMES, THRESHOLD, MAX_LAG):
        meta = dict(video=rec["name"], side=o["side"], cam=rec["cam"], ex=rec["ex"], view=o["view"], person=person_of(rec["name"]))
        for v in ("A", "B"):
            rows[v].append(make_row(v, o["preds"][v], o["truth"], meta))
        records_for_curve.append(dict(**meta, raw_A=o["preds"]["A"], raw_B=o["preds"]["B"], truth=o["truth"]))
for v in ("A", "B"):
    save_rows(rows[v], f"results/variant_{v}.csv")
from benchmark_common import load_variants, results_table, to_markdown
df = load_variants("results/variant_*.csv")
print(to_markdown(results_table(df)))

NameError: name 'mine' is not defined

In [7]:
# Ask B for data/variant_C_angles.csv with columns: video,side,frame,angle  (frame = ABSOLUTE video frame number)
from calibration import fit_curve, fit_without_each_person, apply_calibration, leave_one_person_out, save_params
RAW = "raw_A"            # switch to "raw_C" when B's file arrives
FIT_VIEWS = {"front"}    # fit on front-view data first
recs_fit = [dict(person=r["person"], raw=r[RAW], truth=r["truth"]) for r in records_for_curve if r["view"] in FIT_VIEWS]
print(len(recs_fit), "recordings used to fit;", len({r["person"] for r in recs_fit}), "people")
lopo = leave_one_person_out(recs_fit)
print(lopo.round(1).to_string()); print(lopo.median(numeric_only=True).round(1))
# range_cal much smaller than range_truth = the curve is squeezing range of motion = fail

held = fit_without_each_person(recs_fit)
final = fit_curve(np.concatenate([r["raw"] for r in recs_fit]), np.concatenate([r["truth"] for r in recs_fit]))
rows_D = []
for r in records_for_curve:
    p = held.get(r["person"], final)
    meta = {k: r[k] for k in ("video", "side", "cam", "ex", "view", "person")}
    rows_D.append(make_row("D", apply_calibration(r[RAW], p), r["truth"], meta))
save_rows(rows_D, "results/variant_D.csv")
final["variant"] = RAW[-1]
save_params("data/population_curve.json", final)     # calibrate_ui.py and demo_cv.py read this
print(to_markdown(results_table(load_variants("results/variant_*.csv"))))

0 recordings used to fit; 0 people
Empty DataFrame
Columns: []
Index: []
Series([], dtype: float64)


ValueError: need at least one array to concatenate